<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

```
Input: ψ_soil (soil water supply)
       E (transpiration rate from energy balance)
       LSC (soil-to-leaf conductance from soil_resistance)
       h (leaf height)
       C (plant capacitance)
       ψ_leaf_old (current leaf water potential)
       dt (time step)
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  Gravitational head                  │
    │  head = ρ_water · g  (MPa/m)         │
    └──────────────────────────────────────┘
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  Steady-state target                 │
    │  a = ψ_soil − head·h − E/LSC         │
    │      ↑          ↑         ↑          │
    │    supply    gravity   transpiration │
    │    from      penalty   drawdown      │
    │    soil                              │
    └──────────────────────────────────────┘
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  Time constant                       │
    │  τ = C / LSC  (seconds)              │
    └──────────────────────────────────────┘
                    │
                    ▼
    ┌──────────────────────────────────────┐
    │  Analytical integration              │
    │  Δψ = (a − ψ₀) · (1 − e^(−dt/τ))     │
    │  ψ_leaf_new = ψ₀ + Δψ                │
    └──────────────────────────────────────┘
                    │
                    ▼
            Return ψ_leaf_new
                    │
          (used by StomataEfficiency
           to check: ψ_leaf > ψ_min ?)
```

In [0]:
#| echo: false
#| output: asis
show_doc(leaf_water_potential)

---

[source](https://github.com/ecamo19/plant_hydraulics/blob/main/plant_hydraulics/leaf_water_potential.py#L109){target="_blank" style="float:right; font-size:smaller"}

### leaf_water_potential

```python

def leaf_water_potential(
    physcon, # Physical constants (denh2o, grav).
    leaf, # Must always have:  capac, minl_wp
For Mode A also:   vc_b, vc_c, gplant
For Mode B also:   (uses flux.lsc, no extra leaf params)
    flux, # etflx   — transpiration [mol H₂O/m²/s]
psi_leaf — water potential at previous timestep [MPa]
psi_soil — soil water potential [MPa]
lsc      — leaf-specific conductance [mmol/m²/s/MPa]
height   — canopy height [m]
dt       — timestep [s]
):


```

*Leaf water potential with supply-curve equilibrium + capacitance dynamics.*

The computation has two independent stages:

- Stage 1: Equilibrium target (ψ_eq):

    Where ψ_leaf WOULD settle if given infinite time.

    - Mode A (supply curve, when leaf.vc_b and leaf.vc_c are set):
        Invert  E = ∫[ψ_leaf→ψ_soil] k_max · f(ψ) dψ
        Correctly accounts for nonlinear conductance loss (cavitation).
        Faithful to Sperry 2017.

    - Mode B (Ohm's law fallback, original behaviour):
        ψ_eq = ψ_soil − gravity − E / lsc
        Assumes constant conductance. Used by Medlyn / WUE models
        which don't set vc_b / vc_c.

- Stage 2 — Capacitance dynamics (Bonan Eq. 13.11):
    ψ_leaf moves toward ψ_eq exponentially:
        τ      = leaf.capac / flux.lsc          [time constant, s]
        dy     = (ψ_eq − ψ₀) × (1 − e^{−dt/τ})
        ψ_leaf = ψ₀ + dy

    When dt >> τ:  ψ_leaf → ψ_eq  (quasi-steady-state, Sperry 2017)
    When dt << τ:  ψ_leaf ≈ ψ₀    (capacitance buffers the change)

    Solves the plant capacitance equation for the change in leaf water
    potential over a model time step. The leaf water potential is driven
    toward a steady-state value set by the balance between soil water
    supply and transpiration demand, with the rate of approach governed
    by the plant capacitance.

    The steady-state target is:

        a = ψ_soil - ρ·g·h - E / LSC

    where the three terms represent soil water potential (i.e Pre-dawn), the
    gravitational head (potential energy by unit of weight) for lifting water
    to leaf height, and the transpiration-driven drawdown through the
    soil-to-leaf pathway.

    The change in ψ_leaf is integrated analytically as:

        Δψ = (a - ψ_leaf) · (1 - exp(-dt / τ))

    where τ = C / LSC is the time constant (plant capacitance divided
    by leaf-specific conductance).

    The governing ODE is dy/dt = (a - y) / τ, where y = ψ_leaf,
    a is the steady-state target, and τ = C / LSC. This is integrated
    exactly over the time step dt to give:

        dy = (a - y) · (1 - exp(-dt / τ))

    The factor 1000 in the transpiration term converts flux.etflx
    from mol H2O/m2/s to mmol H2O/m2/s for consistency with LSC
    units.

    The behaviour:

    - If dt ≪ τ (short time step relative to the time constant):
      the exponential ≈ 1 − dt/τ, so Δψ ≈ (a − ψ₀) · dt/τ — a small linear step
      toward the target

    - If dt ≫ τ (long time step): the exponential → 0, so Δψ ≈ a − ψ₀ — the leaf
      jumps all the way to the steady state

    - If ψ₀ = a already (leaf is at equilibrium): Δψ = 0 — nothing changes

    __The bathtub analogy applied to dy = (a - y) · (1 - exp(-dt / τ))__

    A leaf can be viewed as a bathtub where the inflow represents the water
    supply from the soil and the outflow represents the transpiration.
    In this analogy the water potential is the bathtub water level. In the
    absence of capacitance, what comes in immediately comes out (Eq 13.6).
    However, to represent the effect of capacitance over the water potential,
    a tau_b term is incorporated (Eq 13.11). tau_b is the ratio between bathtub
    size and pipe water supply (capacitance / leaf-specific conductance) and
    represents the time it takes the water level to change. Therefore, if the
    bathtub is big (large capacitance) and the water supply pipe is small,
    then the ratio will be large, indicating that it takes a lot of time to
    change the water level (water potential).

    In other words tau_b it is the ratio that determines the respose time for the
    change in water potential

#### Example leaf_water_potential()